In [1]:
import importlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

pd.set_option('display.max_rows', None)  
pd.set_option('display.max_columns', None)

In [2]:
# ─── League Selection ─────────────────────────────────────────────
LEAGUE = 'srb'  # 'cg' = Montenegro 1. CFL | 'srb' = Serbian SuperLiga

# ─── Path Configuration ──────────────────────────────────────────
DATA_DIR = Path("..") / "data"
RAW_DATA_DIR = DATA_DIR / "raw" / LEAGUE
PROCESSED_DATA_DIR = DATA_DIR / "processed" / LEAGUE

OUT_DIR = RAW_DATA_DIR
RAW_DIR = OUT_DIR / "raw_by_match"
PLAYER_PHOTOS_DIR = PROCESSED_DATA_DIR / "player_photos"
TEAM_LOGOS_DIR = PROCESSED_DATA_DIR / "team_logos"

# Create directories if they don't exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PLAYER_PHOTOS_DIR.mkdir(parents=True, exist_ok=True)
TEAM_LOGOS_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# Load matches overview
matches_df = pd.read_csv(OUT_DIR / "matches.csv")

# Keep only essential columns
columns_to_keep = [
    'id', 'season.name', 'status.description', 'roundInfo.round', 
    'homeTeam.name', 'homeTeam.id', 'awayTeam.name', 'awayTeam.id',
    'homeScore.period1', 'homeScore.period2', 'awayScore.period1', 'awayScore.period2',
    'startTimestamp'
]

cols_to_keep = [col for col in columns_to_keep if col in matches_df.columns]
matches_df_clean = matches_df[cols_to_keep].copy()
matches_df_clean = matches_df_clean[matches_df_clean['status.description'] != 'Postponed']

print(f"Total matches (after removing postponed): {len(matches_df_clean)}")

Total matches (after removing postponed): 224


In [4]:
# Create team ID to name mapping from matches
team_mapping = {}
for _, row in matches_df_clean.iterrows():
    team_mapping[row['homeTeam.id']] = row['homeTeam.name']
    team_mapping[row['awayTeam.id']] = row['awayTeam.name']

# Create set of valid match IDs (excluding postponed matches)
valid_match_ids = set(matches_df_clean['id'].astype(str))
print(f"✅ Valid match IDs: {len(valid_match_ids)} matches (postponed matches excluded)")

# Load player match statistics from lineups.json files
player_stats_list = []
processed_matches = 0
for match_dir in RAW_DIR.glob("*"):
    # Skip postponed matches
    if match_dir.name not in valid_match_ids:
        continue
    
    lineups_file = match_dir / "lineups.json"
    if lineups_file.exists():
        try:
            with open(lineups_file, encoding='utf-8') as f:
                lineups_data = json.load(f)
            
            processed_matches += 1
            
            # Extract player statistics from both home and away teams
            for team_side in ['home', 'away']:
                if team_side in lineups_data:
                    team_data = lineups_data[team_side]
                    players = team_data.get('players', [])
                    
                    for player_entry in players:
                        player_info = player_entry.get('player', {})
                        stats = player_entry.get('statistics', {})
                        team_id = player_entry.get('teamId')
                        team_name = team_mapping.get(team_id, 'Unknown')
                        
                        # Combine player info with their statistics and team info
                        record = {
                            'match_id': match_dir.name,
                            'team_side': team_side,
                            'team_id': team_id,
                            'team_name': team_name,
                            'player_id': player_info.get('id'),
                            'player_name': player_info.get('name'),
                            'shortName': player_info.get('shortName'),
                            'position': player_info.get('position'),
                            'jerseyNumber': player_info.get('jerseyNumber'),
                            'substitute': player_entry.get('substitute', False),
                            **stats  # Unpack all statistics
                        }
                        player_stats_list.append(record)
        
        except Exception as e:
            print(f"Could not process {lineups_file}: {e}")

player_stats_df = pd.DataFrame(player_stats_list)
print(f"✅ Player stats loaded: {len(player_stats_df)} player records from {processed_matches} matches")
print(f"\n📊 Unique players: {player_stats_df['player_id'].nunique()}")
print(f"📊 Available statistics columns: {len(player_stats_df.columns)}")

✅ Valid match IDs: 224 matches (postponed matches excluded)
✅ Player stats loaded: 9693 player records from 224 matches

📊 Unique players: 600
📊 Available statistics columns: 83


In [5]:

# =============================================================================
# Scrape player current teams from Sofascore API
# =============================================================================

def get_player_current_team(player_id: int, driver) -> dict:
    """Fetch player's current team and preferred foot from Sofascore API."""
    try:
        url = f"https://api.sofascore.com/api/v1/player/{player_id}"
        driver.get(url)
        time.sleep(0.3)  # Small delay to avoid rate limiting
        pre_tag = driver.find_element(By.TAG_NAME, "pre")
        data = json.loads(pre_tag.text)
        
        player_data = data.get('player', {})
        team = player_data.get('team', {})
        
        return {
            'player_id': player_id,
            'current_team_id': team.get('id'),
            'current_team_name': team.get('name'),
            'retired': not bool(team.get('id')),  # If no team, likely retired
            'preferredFoot': player_data.get('preferredFoot'),
        }
    except Exception as e:
        print(f"Error fetching player {player_id}: {e}")
        return {
            'player_id': player_id,
            'current_team_id': None,
            'current_team_name': None,
            'retired': None,
            'preferredFoot': None,
        }

# Check if we already have cached player team data
player_teams_cache_file = PROCESSED_DATA_DIR / "player_current_teams.json"

if player_teams_cache_file.exists():
    print("📂 Loading cached player team data...")
    with open(player_teams_cache_file, 'r', encoding='utf-8') as f:
        player_current_teams = json.load(f)
    player_current_teams = {int(k): v for k, v in player_current_teams.items()}
    print(f"✅ Loaded {len(player_current_teams)} player team mappings from cache")

    # Check for new players OR players missing preferredFoot data
    all_player_ids = set(player_stats_df['player_id'].dropna().astype(int).unique())
    missing_ids = all_player_ids - set(player_current_teams.keys())
    missing_foot = {
        pid for pid, data in player_current_teams.items()
        if pid in all_player_ids and 'preferredFoot' not in data
    }
    ids_to_fetch = missing_ids | missing_foot

    if ids_to_fetch:
        reason_parts = []
        if missing_ids:
            reason_parts.append(f"{len(missing_ids)} new players")
        if missing_foot:
            reason_parts.append(f"{len(missing_foot)} players missing preferredFoot")
        print(f"\n🆕 Found {', '.join(reason_parts)} — fetching from API...")
        options = webdriver.ChromeOptions()
        options.add_argument("--headless")
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        driver_temp = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
        try:
            for player_id in tqdm(sorted(ids_to_fetch), desc="Fetching player data"):
                result = get_player_current_team(int(player_id), driver_temp)
                player_current_teams[int(player_id)] = result
        finally:
            driver_temp.quit()

        # Save updated cache
        with open(player_teams_cache_file, 'w', encoding='utf-8') as f:
            json.dump(player_current_teams, f, indent=2)
        print(f"✅ Cache updated — now contains {len(player_current_teams)} players")
    else:
        print("   No new players to fetch.")
else:
    print("🌐 Scraping player current teams from Sofascore...")
    print("   This will take a few minutes for all players...")
    
    # Initialize Selenium WebDriver
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
    
    try:
        unique_player_ids = player_stats_df['player_id'].unique()
        player_current_teams = {}
        
        for player_id in tqdm(unique_player_ids, desc="Fetching player teams"):
            result = get_player_current_team(int(player_id), driver)
            player_current_teams[int(player_id)] = result
        
        # Save to cache
        with open(player_teams_cache_file, 'w', encoding='utf-8') as f:
            json.dump(player_current_teams, f, indent=2)
        
        print(f"✅ Scraped and cached {len(player_current_teams)} player team mappings")
        
    finally:
        driver.quit()
        print("🔒 Closed browser")

# Create simple mappings for easy lookup
player_to_current_team = {
    pid: data.get('current_team_name') 
    for pid, data in player_current_teams.items()
}

player_to_current_team_id = {
    pid: data.get('current_team_id') 
    for pid, data in player_current_teams.items()
}

player_to_preferred_foot = {
    pid: data.get('preferredFoot')
    for pid, data in player_current_teams.items()
}

print(f"\n📊 Team status summary:")
retired_count = sum(1 for data in player_current_teams.values() if data.get('retired'))
active_count = sum(1 for data in player_current_teams.values() if not data.get('retired') and data.get('current_team_name'))
unknown_count = sum(1 for data in player_current_teams.values() if data.get('current_team_name') is None and not data.get('retired'))
print(f"   - Active players: {active_count}")
print(f"   - Retired/No team: {retired_count}")
print(f"   - Unknown status: {unknown_count}")

foot_counts = pd.Series(player_to_preferred_foot).value_counts()
print(f"\n📊 Preferred foot distribution:")
for foot, count in foot_counts.items():
    print(f"   - {foot}: {count}")


📂 Loading cached player team data...
✅ Loaded 600 player team mappings from cache
   No new players to fetch.

📊 Team status summary:
   - Active players: 600
   - Retired/No team: 0
   - Unknown status: 0

📊 Preferred foot distribution:
   - Right: 398
   - Left: 127
   - Both: 45


## Player Performance Analysis

Analyze player performance using actual minutes played and comprehensive statistics from lineups.json.

In [6]:
# Top 20 players by total minutes played
player_minutes_summary = player_stats_df.groupby(['player_id', 'player_name', 'position']).agg({
    'minutesPlayed': 'sum',
    'match_id': 'count',  # Number of match appearances
    'goals': 'sum',
    'goalAssist': 'sum',
    'rating': 'mean'
}).reset_index()

player_minutes_summary.columns = ['player_id', 'player_name', 'position', 'total_minutes', 'matches', 'goals', 'assists', 'avg_rating']
player_minutes_summary = player_minutes_summary.sort_values('total_minutes', ascending=False)
player_minutes_summary['avg_rating'] = player_minutes_summary['avg_rating'].round(2)

print(f"Total unique players: {len(player_minutes_summary)}")
print("=== TOP 10 PLAYERS BY MINUTES PLAYED ===\n")
display(player_minutes_summary.head(10))

Total unique players: 599
=== TOP 10 PLAYERS BY MINUTES PLAYED ===



,player_id,player_name,position,total_minutes,matches,goals,assists,avg_rating
58,267437,Zoran Popović,G,2520.0,28,0.0,0.0,7.14
133,844128,Dragan Rosić,G,2520.0,28,0.0,0.0,7.06
416,1427163,Nikola Simic,D,2510.0,28,1.0,0.0,6.97
368,1165955,Dušan Pavlović,D,2430.0,27,2.0,1.0,6.99
126,834186,Nikola Vasiljević,G,2430.0,28,0.0,0.0,6.76
141,862782,Lazar Nikolić,D,2430.0,27,1.0,1.0,7.22
337,1147541,Mateja Gašić,D,2427.0,27,1.0,4.0,7.07
105,799179,Nemanja Miletić,D,2426.0,27,0.0,3.0,6.99
6,38583,Saša Stamenković,G,2413.0,27,0.0,0.0,7.35
432,1468253,Đorđe Skoko,D,2384.0,27,0.0,1.0,6.92


## Organized Data Export

This section creates a clean, organized data structure with:
1. **players_metadata.csv** - Static player info (name, position, height, country, etc.)
2. **teams_metadata.csv** - Static team info (name, colors, country)
3. **matches_metadata.csv** - Match info (teams, scores, timestamp, round)
4. **match_player_statistics.csv** - Per-player per-match stats
5. **match_team_statistics.csv** - Per-team per-match stats

This eliminates the need to iterate over raw data for common analysis tasks.

In [7]:

# =============================================================================
# 1. TEAMS METADATA - Extract static team information
# =============================================================================

import sys
sys.path.insert(0, str(Path('..') / 'src'))
from config import LEAGUES
TEAM_SHORT_NAMES = LEAGUES[LEAGUE].get('team_short_names', {})

teams_data = {}

# Extract from matches
for _, row in matches_df.iterrows():
    # Home team
    if row['homeTeam.id'] not in teams_data:
        teams_data[row['homeTeam.id']] = {
            'team_id': row['homeTeam.id'],
            'team_name': row['homeTeam.name'],
            'short_name': TEAM_SHORT_NAMES.get(row['homeTeam.id'], row['homeTeam.name'])
        }
    # Away team
    if row['awayTeam.id'] not in teams_data:
        teams_data[row['awayTeam.id']] = {
            'team_id': row['awayTeam.id'],
            'team_name': row['awayTeam.name'],
            'short_name': TEAM_SHORT_NAMES.get(row['awayTeam.id'], row['awayTeam.name'])
        }

teams_meta = pd.DataFrame(list(teams_data.values()))
teams_output_path = PROCESSED_DATA_DIR / "teams_metadata.csv"
teams_meta.to_csv(teams_output_path, index=False)

print(f"✅ Saved {len(teams_meta)} teams to {teams_output_path}")
display(teams_meta)


✅ Saved 16 teams to ..\data\processed\srb\teams_metadata.csv


,team_id,team_name,short_name
0,282203,FK Železničar Pančevo,Železničar
1,5150,FK Vojvodina,Vojvodina
2,5152,FK Partizan,Partizan
3,5153,OFK Beograd,OFK Beograd
4,25737,FK Radnički Niš,Radnički Niš
5,5144,FK Javor Ivanjica,Javor
6,7674,FK Napredak Kruševac,Napredak
7,36967,FK Spartak Subotica,Spartak
8,43981,FK Radnički 1923,Radnički 1923
9,5149,FK Crvena zvezda,Crvena zvezda


In [8]:
# =============================================================================
# 2. PLAYERS METADATA - Extract STATIC player information (no match stats)
# =============================================================================

players_static_data = {}

for match_dir in tqdm(RAW_DIR.glob("*"), desc="Extracting player metadata"):
    if match_dir.name not in valid_match_ids:
        continue
    
    lineups_file = match_dir / "lineups.json"
    if lineups_file.exists():
        try:
            with open(lineups_file, encoding='utf-8') as f:
                lineups_data = json.load(f)
            
            for team_side in ['home', 'away']:
                if team_side in lineups_data:
                    for player_entry in lineups_data[team_side].get('players', []):
                        player_info = player_entry.get('player', {})
                        player_id = player_info.get('id')
                        
                        if player_id and player_id not in players_static_data:
                            # Extract country info from nested structure
                            country = player_info.get('country', {})
                            
                            # Extract market value from proposedMarketValueRaw
                            proposed_market_value = player_info.get('proposedMarketValueRaw', {})
                            market_value = proposed_market_value.get('value') if proposed_market_value else None
                            market_value_currency = proposed_market_value.get('currency', '') if proposed_market_value else player_info.get('marketValueCurrency', '')
                            
                            players_static_data[player_id] = {
                                'id': player_id,
                                'name': player_info.get('name', ''),
                                'shortName': player_info.get('shortName', ''),
                                'position': player_info.get('position', ''),
                                'height': player_info.get('height'),
                                'country_name': country.get('name', '') if country else '',
                                'dateOfBirthTimestamp': player_info.get('dateOfBirthTimestamp'),
                                'marketValue': market_value
                            }
        except Exception as e:
            print(f"Could not process {lineups_file}: {e}")

# Convert to DataFrame
players_static = pd.DataFrame(list(players_static_data.values()))

# Convert timestamp to readable date
def timestamp_to_date(ts):
    if pd.isna(ts) or ts is None:
        return None
    try:
        return pd.to_datetime(ts, unit='s').strftime('%Y-%m-%d')
    except:
        return None

players_static['dateOfBirth'] = players_static['dateOfBirthTimestamp'].apply(timestamp_to_date)
players_static = players_static.drop(columns=['dateOfBirthTimestamp'])

# Add preferred foot from cached player API data
players_static['preferredFoot'] = players_static['id'].map(player_to_preferred_foot)

# Add has_player_photo flag
players_static['has_player_photo'] = players_static['id'].apply(
    lambda pid: (PLAYER_PHOTOS_DIR / f"{pid}.png").exists()
)

# Save
players_output_path = PROCESSED_DATA_DIR / "players_metadata.csv"
players_static.to_csv(players_output_path, index=False)

print(f"\n✅ Saved {len(players_static)} players to {players_output_path}")
display(players_static.head(10))

Extracting player metadata: 229it [00:00, 2316.16it/s]


✅ Saved 600 players to ..\data\processed\srb\players_metadata.csv


,id,name,shortName,position,height,country_name,marketValue,dateOfBirth,preferredFoot,has_player_photo
0,603946,Matheus,Matheus,G,190.0,Brazil,2300000.0,1992-07-19,Right,True
1,1019333,Young-woo Seol,Y. W. Seol,D,181.0,South Korea,4600000.0,1998-12-05,Right,True
2,1406130,Veljko Milosavljević,V. Milosavljević,D,192.0,Serbia,18600000.0,2007-06-28,Right,True
3,227224,Miloš Veljković,M. Veljković,D,188.0,Serbia,2300000.0,1995-09-26,Right,True
4,904400,Nair Tiknizyan,N. Tiknizyan,D,180.0,Armenia,4099999.0,1999-05-12,Left,True
5,155449,Rade Krunić,R. Krunić,M,184.0,Bosnia & Herzegovina,3800000.0,1993-10-07,Right,True
6,1153956,Jovan Šljivić,J. Šljivić,M,176.0,Serbia,960000.0,2005-10-14,Right,True
7,939721,Milson,Milson,M,170.0,Angola,3600000.0,1999-10-12,Right,True
8,110701,Aleksandar Katai,A. Katai,M,182.0,Serbia,725000.0,1991-02-06,Right,True
9,278763,Mirko Ivanić,M. Ivanić,M,182.0,Montenegro,3800000.0,1993-09-13,Both,True


In [9]:
# =============================================================================
# 3. MATCHES METADATA - Clean match information
# =============================================================================

matches_meta = matches_df_clean[['id', 'season.name', 'status.description', 'roundInfo.round',
                                 'homeTeam.name', 'homeTeam.id', 'awayTeam.name', 'awayTeam.id',
                                 'homeScore.period1', 'homeScore.period2', 'awayScore.period1', 'awayScore.period2',
                                 'startTimestamp']].copy()

# Rename columns for clarity
matches_meta.columns = [
    'match_id', 'season', 'status', 'round',
    'homeTeam_name', 'homeTeam_id', 'awayTeam_name', 'awayTeam_id',
    'homeScore_period1', 'homeScore_period2', 'awayScore_period1', 'awayScore_period2',
    'startTimestamp'
]

# Add total scores
matches_meta['homeScore_total'] = matches_meta['homeScore_period1'].fillna(0) + matches_meta['homeScore_period2'].fillna(0)
matches_meta['awayScore_total'] = matches_meta['awayScore_period1'].fillna(0) + matches_meta['awayScore_period2'].fillna(0)

# Convert timestamp to readable datetime
matches_meta['match_datetime'] = pd.to_datetime(matches_meta['startTimestamp'], unit='s')
matches_meta['match_date'] = matches_meta['match_datetime'].dt.strftime('%Y-%m-%d')

# Define the set of matches we expect formations for (postponed already removed)
matches_with_lineups = set(matches_meta['match_id'].astype(str))

# Collect formations from lineups (skip postponed matches already filtered out)
formation_records = []
missing_formations = []

for match_dir in tqdm(RAW_DIR.glob("*"), desc="Collecting formations"):
    match_id = match_dir.name
    if match_id not in matches_with_lineups:
        continue

    lineups_file = match_dir / "lineups.json"
    if not lineups_file.exists():
        continue

    try:
        with open(lineups_file, encoding='utf-8') as f:
            lineups_data = json.load(f)
    except Exception as e:
        print(f"⚠️ Could not read formations for match {match_id}: {e}")
        continue

    home_formation = lineups_data.get('home', {}).get('formation')
    away_formation = lineups_data.get('away', {}).get('formation')

    if not home_formation or not away_formation:
        missing_formations.append(int(match_id))
        continue

    formation_records.append({
        'match_id': int(match_id),
        'homeTeam_formation': home_formation,
        'awayTeam_formation': away_formation,
    })

formations_df = pd.DataFrame(formation_records).drop_duplicates(subset=['match_id'], keep='last')

if formations_df.empty:
    raise ValueError("No formation data collected. Please verify the raw lineups files.")

if missing_formations:
    print(f"⚠️ Skipped {len(set(missing_formations))} matches without formation data: {sorted(set(missing_formations))}")

# Merge formations into metadata
matches_meta = matches_meta.merge(formations_df, on='match_id', how='left')

# Reorder columns
matches_meta = matches_meta[[
    'match_id', 'round', 'match_date', 'match_datetime', 'status',
    'homeTeam_id', 'homeTeam_name', 'homeTeam_formation',
    'awayTeam_id', 'awayTeam_name', 'awayTeam_formation',
    'homeScore_period1', 'homeScore_period2', 'homeScore_total',
    'awayScore_period1', 'awayScore_period2', 'awayScore_total'
]]

# Save
matches_output_path = PROCESSED_DATA_DIR / "matches_metadata.csv"
matches_meta.to_csv(matches_output_path, index=False)

print(f"✅ Saved {len(matches_meta)} matches to {matches_output_path}")
display(matches_meta.head(10))

✅ Saved 224 matches to ..\data\processed\srb\matches_metadata.csv


,match_id,round,match_date,match_datetime,status,homeTeam_id,homeTeam_name,homeTeam_formation,awayTeam_id,awayTeam_name,awayTeam_formation,homeScore_period1,homeScore_period2,homeScore_total,awayScore_period1,awayScore_period2,awayScore_total
0,14015604,25,2026-02-28,2026-02-28 15:00:00,Ended,282203,FK Železničar Pančevo,4-1-4-1,5150,FK Vojvodina,4-2-3-1,2.0,0.0,2.0,0.0,0.0,0.0
1,14015603,25,2026-02-28,2026-02-28 17:30:00,Ended,5152,FK Partizan,4-2-3-1,5153,OFK Beograd,5-3-2,1.0,0.0,1.0,1.0,1.0,2.0
2,14015599,25,2026-03-01,2026-03-01 12:00:00,Ended,25737,FK Radnički Niš,4-1-4-1,5144,FK Javor Ivanjica,4-2-3-1,0.0,1.0,1.0,0.0,0.0,0.0
3,14015601,25,2026-03-01,2026-03-01 14:30:00,Ended,7674,FK Napredak Kruševac,4-2-3-1,36967,FK Spartak Subotica,5-4-1,0.0,1.0,1.0,0.0,1.0,1.0
4,14015602,25,2026-03-01,2026-03-01 16:00:00,Ended,43981,FK Radnički 1923,4-2-3-1,5149,FK Crvena zvezda,4-2-3-1,0.0,0.0,0.0,1.0,1.0,2.0
5,14015600,25,2026-03-02,2026-03-02 16:00:00,Ended,308176,FK IMT Beograd,4-1-4-1,111127,FK Radnik Surdulica,3-1-4-2,0.0,0.0,0.0,0.0,0.0,0.0
6,14015609,26,2026-03-07,2026-03-07 12:00:00,Ended,263360,FK TSC Bačka Topola,4-2-3-1,282203,FK Železničar Pančevo,4-1-4-1,0.0,0.0,0.0,0.0,0.0,0.0
7,14015607,26,2026-03-07,2026-03-07 14:00:00,Ended,5144,FK Javor Ivanjica,4-2-3-1,5148,FK Čukarički,4-2-3-1,1.0,0.0,1.0,0.0,0.0,0.0
8,14015614,26,2026-03-08,2026-03-08 13:00:00,Ended,111127,FK Radnik Surdulica,4-2-3-1,25737,FK Radnički Niš,4-1-4-1,0.0,2.0,2.0,1.0,1.0,2.0
9,14015613,26,2026-03-08,2026-03-08 15:00:00,Ended,36967,FK Spartak Subotica,4-4-2,308176,FK IMT Beograd,4-1-4-1,0.0,0.0,0.0,0.0,1.0,1.0


In [10]:
# =============================================================================
# 4. MATCH PLAYER STATISTICS - Complete per-match per-player statistics
# =============================================================================

match_player_stats_list = []
dribbles_list = []

for match_dir in tqdm([d for d in RAW_DIR.glob("*") if d.name in valid_match_ids], desc="Loading player stats"):
    with open(match_dir / "lineups.json", encoding='utf-8') as f:
        lineups_data = json.load(f)
    
    # Get match info for team IDs and names
    match_info = matches_df_clean[matches_df_clean['id'] == int(match_dir.name)].iloc[0]
    team_ids = {
        'home': int(match_info['homeTeam.id']),
        'away': int(match_info['awayTeam.id'])
    }
    team_names = {
        'home': match_info['homeTeam.name'],
        'away': match_info['awayTeam.name']
    }
    
    for team_side in ['home', 'away']:
        team_data = lineups_data[team_side]
        
        for player_entry in team_data['players']:
            player_info = player_entry['player']
            stats = player_entry.get('statistics', {})
            
            record = {
                'match_id': int(match_dir.name),
                'player_id': player_info['id'],
                'player_name': player_info.get('name', ''),
                'team_id': team_ids[team_side],
                'team_name': team_names[team_side],
                'team_side': team_side,
                'position_played': player_entry.get('position', player_info.get('position', '')),
                'shirtNumber': player_entry.get('shirtNumber'),
                'substitute': player_entry.get('substitute', False),
                'played': stats.get('minutesPlayed', 0) > 0,
                **{k: v for k, v in stats.items() if k not in ['ratingVersions', 'statisticsType']}
            }
            match_player_stats_list.append(record)
    
    # Load dribbles data
    stats_tabs_file = match_dir / "player_stats_tabs.csv"
    if stats_tabs_file.exists():
        stats_tabs_df = pd.read_csv(stats_tabs_file)
        dribbles_df = stats_tabs_df[
            (stats_tabs_df['metric_key'] == 'totalContest') & 
            (stats_tabs_df['metric'] == 'Dribbles (successful)')
        ][['match_id', 'player_id', 'value', 'secondary']].rename(
            columns={'value': 'totalDribble', 'secondary': 'successfulDribbles'}
        )
        if len(dribbles_df) > 0:
            dribbles_list.append(dribbles_df)

# Create DataFrame and merge dribbles
match_player_stats = pd.DataFrame(match_player_stats_list)

if dribbles_list:
    dribbles_data = pd.concat(dribbles_list, ignore_index=True).drop_duplicates(subset=['match_id', 'player_id'])
    match_player_stats = match_player_stats.drop(columns=['successfulDribbles', 'totalDribble'], errors='ignore').merge(
        dribbles_data, on=['match_id', 'player_id'], how='left'
    )
    print(f"✅ Merged dribbles data for {len(dribbles_data)} player-match records")

match_player_stats[['successfulDribbles', 'totalDribble']] = match_player_stats[['successfulDribbles', 'totalDribble']].fillna(0)

# Add player's current team from scraped data (both ID and name)
match_player_stats['player_current_team_id'] = match_player_stats['player_id'].map(player_to_current_team_id)
match_player_stats['player_current_team'] = match_player_stats['player_id'].map(player_to_current_team)

# Reorder columns
id_cols = ['match_id', 'player_id', 'player_name', 'team_id', 'team_name', 'player_current_team_id', 'player_current_team', 'team_side', 'position_played', 'shirtNumber', 'substitute', 'played']
stat_cols = [col for col in match_player_stats.columns if col not in id_cols]
match_player_stats = match_player_stats[id_cols + sorted(stat_cols)]

# Save
match_player_stats.to_csv(PROCESSED_DATA_DIR / "match_player_statistics.csv", index=False)

print(f"✅ Saved {len(match_player_stats)} player-match records")
print(f"   - Statistics columns: {len(stat_cols)}")
display(match_player_stats.head(5))

Loading player stats: 100%|██████████| 224/224 [00:01<00:00, 197.35it/s]


✅ Merged dribbles data for 9693 player-match records
✅ Saved 9693 player-match records
   - Statistics columns: 73


,match_id,player_id,player_name,team_id,team_name,player_current_team_id,player_current_team,team_side,position_played,shirtNumber,substitute,played,accurateCross,accurateKeeperSweeper,accurateLongBalls,accurateOppositionHalfPasses,accurateOwnHalfPasses,accuratePass,aerialLost,aerialWon,ballCarriesCount,ballRecovery,bestBallCarryProgression,bigChanceCreated,bigChanceMissed,blockedScoringAttempt,challengeLost,clearanceOffLine,crossNotClaimed,defensiveValueNormalized,dispossessed,dribbleValueNormalized,duelLost,duelWon,errorLeadToAGoal,errorLeadToAShot,fouls,goalAssist,goalkeeperValueNormalized,goals,goodHighClaim,hitWoodwork,interceptionWon,keeperSaveValue,keyPass,lastManTackle,minutesPlayed,onTargetScoringAttempt,outfielderBlock,ownGoals,passValueNormalized,penaltyConceded,penaltyFaced,penaltyMiss,penaltySave,penaltyWon,possessionLostCtrl,progressiveBallCarriesCount,punches,rating,savedShotsFromInsideTheBox,saves,shotOffTarget,shotValueNormalized,successfulDribbles,totalBallCarriesDistance,totalClearance,totalContest,totalCross,totalDribble,totalKeeperSweeper,totalLongBalls,totalOffside,totalOppositionHalfPasses,totalOwnHalfPasses,totalPass,totalProgression,totalProgressiveBallCarriesDistance,totalShots,totalTackle,touches,unsuccessfulTouch,wasFouled,wonContest,wonTackle
0,14015386,603946,Matheus,5149,FK Crvena zvezda,5149,FK Crvena zvezda,home,G,1,False,True,NaN,1.0,2.0,NaN,12.0,12.0,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,7.3,1.0,3.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,1.0,3.0,NaN,NaN,13.0,13.0,NaN,NaN,0,NaN,19.0,NaN,1.0,NaN,NaN
1,14015386,1019333,Young-woo Seol,5149,FK Crvena zvezda,5149,FK Crvena zvezda,home,D,66,False,True,1.0,NaN,1.0,23.0,17.0,40.0,NaN,NaN,NaN,7.0,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,4.0,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,2.0,NaN,2.0,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,8.1,NaN,NaN,1.0,NaN,0.0,NaN,NaN,NaN,3.0,0.0,NaN,2.0,NaN,25.0,19.0,44.0,NaN,NaN,2,1.0,68.0,1.0,3.0,NaN,1.0
2,14015386,1406130,Veljko Milosavljević,5149,FK Crvena zvezda,60,Bournemouth,home,D,44,False,True,NaN,NaN,NaN,51.0,30.0,81.0,4.0,6.0,NaN,4.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,8.0,8.0,NaN,NaN,2.0,0.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN,7.2,NaN,NaN,1.0,NaN,0.0,NaN,3.0,NaN,NaN,0.0,NaN,2.0,NaN,57.0,34.0,91.0,NaN,NaN,1,1.0,101.0,NaN,1.0,NaN,NaN
3,14015386,227224,Miloš Veljković,5149,FK Crvena zvezda,5149,FK Crvena zvezda,home,D,13,False,True,NaN,NaN,NaN,52.0,35.0,87.0,3.0,2.0,NaN,4.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,3.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,7.5,NaN,NaN,1.0,NaN,0.0,NaN,4.0,NaN,NaN,0.0,NaN,2.0,NaN,55.0,36.0,91.0,NaN,NaN,1,1.0,100.0,NaN,NaN,NaN,1.0
4,14015386,904400,Nair Tiknizyan,5149,FK Crvena zvezda,5149,FK Crvena zvezda,home,D,23,False,True,3.0,NaN,1.0,27.0,21.0,48.0,2.0,1.0,NaN,7.0,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,2.0,NaN,8.0,7.0,NaN,NaN,3.0,0.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN,NaN,7.5,NaN,NaN,3.0,NaN,1.0,NaN,2.0,1.0,4.0,1.0,NaN,2.0,1.0,32.0,21.0,53.0,NaN,NaN,3,4.0,80.0,NaN,1.0,1.0,3.0


In [11]:
# =============================================================================
# 5. MATCH TEAM STATISTICS - Team-level per-match statistics from statistics.json
# =============================================================================

match_team_stats_list = []

for match_dir in tqdm(RAW_DIR.glob("*"), desc="Loading team match stats"):
    if match_dir.name not in valid_match_ids:
        continue
    
    stats_file = match_dir / "statistics.json"
    if stats_file.exists():
        try:
            with open(stats_file, encoding='utf-8') as f:
                stats_data = json.load(f)
            
            # Get match info to map team IDs
            match_info = matches_df_clean[matches_df_clean['id'] == int(match_dir.name)]
            if len(match_info) == 0:
                continue
            match_info = match_info.iloc[0]
            
            home_record = {
                'match_id': int(match_dir.name),
                'team_id': int(match_info['homeTeam.id']),
                'team_name': match_info['homeTeam.name'],
                'team_side': 'home',
            }
            away_record = {
                'match_id': int(match_dir.name),
                'team_id': int(match_info['awayTeam.id']),
                'team_name': match_info['awayTeam.name'],
                'team_side': 'away',
            }
            
            # Parse statistics
            for stat_section in stats_data.get('statistics', []):
                if stat_section.get('period') == 'ALL':
                    for group in stat_section.get('groups', []):
                        for item in group.get('statisticsItems', []):
                            key = item.get('key', item.get('name', '').replace(' ', '_').lower())
                            home_record[key] = item.get('homeValue')
                            away_record[key] = item.get('awayValue')
            
            match_team_stats_list.append(home_record)
            match_team_stats_list.append(away_record)
        
        except Exception as e:
            print(f"Could not process {stats_file}: {e}")

# Convert to DataFrame
match_team_stats = pd.DataFrame(match_team_stats_list)

# Reorder columns
id_cols = ['match_id', 'team_id', 'team_name', 'team_side']
stat_cols = [col for col in match_team_stats.columns if col not in id_cols]
match_team_stats = match_team_stats[id_cols + sorted(stat_cols)]

# Save
team_stats_output_path = PROCESSED_DATA_DIR / "match_team_statistics.csv"
match_team_stats.to_csv(team_stats_output_path, index=False)

print(f"✅ Saved {len(match_team_stats)} team-match records to {team_stats_output_path}")
display(match_team_stats.head(10))

Loading team match stats: 229it [00:00, 2068.85it/s]

✅ Saved 448 team-match records to ..\data\processed\srb\match_team_statistics.csv


,match_id,team_id,team_name,team_side,accurateCross,accurateLongBalls,accuratePasses,accurateThroughBall,aerialDuelsPercentage,ballPossession,ballRecovery,bigChanceCreated,bigChanceMissed,bigChanceScored,blockedScoringAttempt,cornerKicks,dispossessed,diveSaves,dribblesPercentage,duelWonPercent,errorsLeadToGoal,errorsLeadToShot,expectedGoals,finalThirdEntries,finalThirdPhaseStatistic,fouledFinalThird,fouls,freeKicks,goalKicks,goalkeeperSaves,groundDuelsPercentage,highClaims,hitWoodwork,interceptionWon,offsides,passes,penaltySaves,punches,redCards,shotsOffGoal,shotsOnGoal,throwIns,totalClearance,totalShotsInsideBox,totalShotsOnGoal,totalShotsOutsideBox,totalTackle,touchesInOppBox,wonTacklePercent,yellowCards
0,14015386,5149,FK Crvena zvezda,home,8,12,525,4.0,13,68,51,8.0,5.0,3.0,5,9,16,1.0,8,50,NaN,NaN,4.32,83,158,5.0,13,11,0,3.0,38,NaN,1,7,2.0,586,NaN,NaN,NaN,15,9,21,9,15,29,14,19,35,11,NaN
1,14015386,5144,FK Javor Ivanjica,away,0,19,208,0.0,12,32,45,0.0,0.0,0.0,0,1,14,1.0,4,50,NaN,NaN,0.12,34,28,0.0,12,13,17,5.0,40,NaN,0,11,0.0,279,NaN,NaN,NaN,2,3,9,28,2,5,3,23,4,12,NaN
2,14015387,5153,OFK Beograd,home,4,23,378,1.0,8,56,42,2.0,1.0,1.0,3,3,11,1.0,8,51,1.0,0.0,0.94,54,65,7.0,13,22,5,4.0,46,0.0,1,10,2.0,444,NaN,1.0,NaN,5,4,16,23,9,12,3,17,15,10,2.0
3,14015387,36967,FK Spartak Subotica,away,3,18,279,2.0,10,44,38,4.0,2.0,2.0,3,8,8,2.0,8,49,0.0,1.0,1.98,51,66,3.0,22,12,7,3.0,41,2.0,1,6,1.0,353,NaN,1.0,NaN,6,7,22,20,11,16,5,20,20,17,3.0
4,14015388,5150,FK Vojvodina,home,3,26,309,2.0,15,48,56,3.0,1.0,2.0,1,3,13,1.0,12,47,0.0,NaN,1.04,57,66,3.0,15,9,7,2.0,42,1.0,0,10,2.0,393,NaN,1.0,1.0,2,5,22,15,5,8,3,22,12,13,3.0
5,14015388,111127,FK Radnik Surdulica,away,1,25,357,1.0,15,52,50,0.0,0.0,0.0,2,2,11,1.0,9,53,1.0,NaN,0.44,52,85,0.0,9,15,3,3.0,49,1.0,0,9,1.0,429,NaN,1.0,0.0,5,2,18,27,2,9,7,26,18,24,2.0
6,14015389,263360,FK TSC Bačka Topola,home,2,26,293,1.0,11,52,52,2.0,NaN,2.0,4,4,11,NaN,7,56,1.0,NaN,1.20,58,78,2.0,13,25,3,4.0,51,NaN,0,10,4.0,373,NaN,1.0,NaN,4,3,22,25,8,11,3,19,24,9,4.0
7,14015389,25737,FK Radnički Niš,away,4,9,246,0.0,14,48,48,0.0,NaN,0.0,0,3,13,NaN,4,44,0.0,NaN,1.10,63,75,2.0,25,13,10,1.0,35,NaN,0,3,4.0,344,NaN,0.0,NaN,3,5,31,14,3,8,5,20,7,14,7.0
8,14015390,7771,FK Mladost Lučani,home,3,24,285,1.0,21,46,46,4.0,3.0,1.0,1,5,8,1.0,7,53,NaN,1.0,1.55,50,65,5.0,12,21,11,3.0,37,0.0,0,12,2.0,368,1.0,0.0,NaN,8,3,22,27,8,12,4,10,18,8,3.0
9,14015390,308176,FK IMT Beograd,away,4,36,340,1.0,18,54,42,4.0,3.0,1.0,6,4,7,0.0,9,47,NaN,0.0,2.13,68,88,2.0,21,11,12,2.0,33,1.0,0,7,0.0,428,0.0,4.0,NaN,3,5,17,24,8,14,6,14,23,6,5.0


In [12]:
# =============================================================================
# 7. BONUS: Aggregated Player Season Statistics (convenience file)
# =============================================================================

# Define desired aggregations — skip columns that don't exist in this league's data
_agg_spec = {
    # Team info (take last/most recent)
    'team_id': 'last',
    'team_name': 'last',
    # Counts
    'match_id': 'count',  # appearances
    # Time
    'minutesPlayed': 'sum',
    # Attacking
    'goals': 'sum',
    'goalAssist': 'sum',
    'expectedGoals': 'sum',
    'expectedAssists': 'sum',
    'totalShots': 'sum',
    'onTargetScoringAttempt': 'sum',
    'keyPass': 'sum',
    'bigChanceCreated': 'sum',
    # Passing
    'totalPass': 'sum',
    'accuratePass': 'sum',
    'totalLongBalls': 'sum',
    'accurateLongBalls': 'sum',
    'totalCross': 'sum',
    'accurateCross': 'sum',
    # Defending
    'totalTackle': 'sum',
    'wonTackle': 'sum',
    'interceptionWon': 'sum',
    'totalClearance': 'sum',
    'ballRecovery': 'sum',
    'outfielderBlock': 'sum',
    # Duels
    'duelWon': 'sum',
    'duelLost': 'sum',
    'aerialWon': 'sum',
    'aerialLost': 'sum',
    # Discipline
    'fouls': 'sum',
    'wasFouled': 'sum',
    'successfulDribbles': 'sum',
    'totalDribble': 'sum',
    # Rating
    'rating': 'mean',
}

# Keep only columns that exist in the data
played = match_player_stats[match_player_stats['played'] == True]
agg_spec = {k: v for k, v in _agg_spec.items() if k in played.columns}

player_season_stats = played.groupby('player_id').agg(agg_spec).reset_index()

# Build rename mapping (order must match agg_spec)
_rename_map = {
    'player_id': 'player_id', 'team_id': 'team_id', 'team_name': 'team_name',
    'match_id': 'appearances', 'minutesPlayed': 'minutes_played',
    'goals': 'goals', 'goalAssist': 'assists',
    'expectedGoals': 'xG', 'expectedAssists': 'xA',
    'totalShots': 'total_shots', 'onTargetScoringAttempt': 'shots_on_target',
    'keyPass': 'key_passes', 'bigChanceCreated': 'big_chances_created',
    'totalPass': 'total_passes', 'accuratePass': 'accurate_passes',
    'totalLongBalls': 'total_long_balls', 'accurateLongBalls': 'accurate_long_balls',
    'totalCross': 'total_crosses', 'accurateCross': 'accurate_crosses',
    'totalTackle': 'total_tackles', 'wonTackle': 'tackles_won',
    'interceptionWon': 'interceptions', 'totalClearance': 'clearances',
    'ballRecovery': 'ball_recoveries', 'outfielderBlock': 'blocks',
    'duelWon': 'duels_won', 'duelLost': 'duels_lost',
    'aerialWon': 'aerial_won', 'aerialLost': 'aerial_lost',
    'fouls': 'fouls_committed', 'wasFouled': 'fouls_won',
    'successfulDribbles': 'dribbles_successful', 'totalDribble': 'dribbles_total',
    'rating': 'avg_rating',
}
player_season_stats = player_season_stats.rename(columns=_rename_map)

# Add xG/xA columns as 0 if they weren't in the data
for col in ['xG', 'xA']:
    if col not in player_season_stats.columns:
        player_season_stats[col] = 0.0

# Add derived metrics
player_season_stats['minutes_per_game'] = (player_season_stats['minutes_played'] / player_season_stats['appearances']).round(1)
player_season_stats['pass_accuracy'] = ((player_season_stats['accurate_passes'] / player_season_stats['total_passes']) * 100).round(1)
player_season_stats['shot_accuracy'] = ((player_season_stats['shots_on_target'] / player_season_stats['total_shots'].replace(0, np.nan)) * 100).round(1)
player_season_stats['dribble_success_rate'] = ((player_season_stats['dribbles_successful'] / player_season_stats['dribbles_total'].replace(0, np.nan)) * 100).round(1)
player_season_stats['duel_success_rate'] = ((player_season_stats['duels_won'] / (player_season_stats['duels_won'] + player_season_stats['duels_lost']).replace(0, np.nan)) * 100).round(1)
player_season_stats['goals_per_90'] = ((player_season_stats['goals'] / player_season_stats['minutes_played']) * 90).round(2)
player_season_stats['assists_per_90'] = ((player_season_stats['assists'] / player_season_stats['minutes_played']) * 90).round(2)
player_season_stats['avg_rating'] = player_season_stats['avg_rating'].round(2)

# Merge with static player info
player_season_stats = player_season_stats.merge(
    players_static[['id', 'name', 'shortName', 'position', 'country_name', 'dateOfBirth', 'has_player_photo']],
    left_on='player_id',
    right_on='id',
    how='left'
).drop(columns=['id'])

# Reorder columns
first_cols = ['player_id', 'name', 'shortName', 'position', 'team_name', 'country_name', 'dateOfBirth']
other_cols = [c for c in player_season_stats.columns if c not in first_cols]
player_season_stats = player_season_stats[first_cols + other_cols]

# Sort by minutes played
player_season_stats = player_season_stats.sort_values('minutes_played', ascending=False).reset_index(drop=True)

# Save
season_stats_output_path = PROCESSED_DATA_DIR / "player_season_statistics.csv"
player_season_stats.to_csv(season_stats_output_path, index=False)

print(f"\u2705 Saved {len(player_season_stats)} aggregated player records to {season_stats_output_path}")
print(f"\n\U0001f3c6 Top 10 players by minutes played:")
player_season_stats.head(5)


✅ Saved 523 aggregated player records to ..\data\processed\srb\player_season_statistics.csv

🏆 Top 10 players by minutes played:


,player_id,name,shortName,position,team_name,country_name,dateOfBirth,team_id,appearances,minutes_played,goals,assists,total_shots,shots_on_target,key_passes,big_chances_created,total_passes,accurate_passes,total_long_balls,accurate_long_balls,total_crosses,accurate_crosses,total_tackles,tackles_won,interceptions,clearances,ball_recoveries,blocks,duels_won,duels_lost,aerial_won,aerial_lost,fouls_committed,fouls_won,dribbles_successful,dribbles_total,avg_rating,xG,xA,minutes_per_game,pass_accuracy,shot_accuracy,dribble_success_rate,duel_success_rate,goals_per_90,assists_per_90,has_player_photo
0,267437,Zoran Popović,Z. Popović,G,FK Železničar Pančevo,Serbia,1988-05-28,282203,28,2520.0,0.0,0.0,0,0.0,0.0,0.0,949.0,600.0,522.0,190.0,0.0,0.0,1.0,1.0,0.0,40.0,275.0,0.0,17.0,2.0,6.0,2.0,0.0,8.0,2.0,2.0,7.14,0.0,0.0,90.0,63.2,NaN,100.0,89.5,0.00,0.00,True
1,844128,Dragan Rosić,D. Rosić,G,FK Vojvodina,Serbia,1996-09-22,5150,28,2520.0,0.0,0.0,0,0.0,0.0,0.0,550.0,305.0,349.0,106.0,0.0,0.0,0.0,0.0,0.0,31.0,231.0,0.0,6.0,0.0,3.0,0.0,0.0,2.0,0.0,0.0,7.06,0.0,0.0,90.0,55.5,NaN,NaN,100.0,0.00,0.00,True
2,1427163,Nikola Simic,N. Simić,D,FK Partizan,Serbia,2007-03-30,5152,28,2510.0,1.0,0.0,12,3.0,3.0,0.0,1695.0,1519.0,149.0,76.0,0.0,0.0,25.0,14.0,24.0,104.0,95.0,12.0,109.0,60.0,61.0,29.0,22.0,20.0,3.0,3.0,6.97,0.0,0.0,89.6,89.6,25.0,100.0,64.5,0.04,0.00,True
3,1165955,Dušan Pavlović,D. Pavlović,D,FK Radnički Niš,Serbia,2004-05-19,25737,27,2430.0,2.0,1.0,18,8.0,7.0,2.0,1371.0,1149.0,196.0,99.0,4.0,1.0,20.0,14.0,31.0,173.0,97.0,26.0,108.0,78.0,61.0,42.0,28.0,23.0,4.0,6.0,6.99,0.0,0.0,90.0,83.8,44.4,66.7,58.1,0.07,0.04,True
4,862782,Lazar Nikolić,L. Nikolić,D,FK Vojvodina,Serbia,1999-08-01,5150,27,2430.0,1.0,1.0,12,1.0,29.0,1.0,1087.0,813.0,181.0,68.0,89.0,20.0,75.0,51.0,33.0,97.0,148.0,9.0,126.0,106.0,23.0,22.0,33.0,19.0,9.0,22.0,7.22,0.0,0.0,90.0,74.8,8.3,40.9,54.3,0.04,0.04,True
